# "Hello World" example with Prefect+Dask

See the associated:

  * Python module: [hello_world_flow.py](./hello_world_flow.py)
  * YAML file: [hello_world_flow.yaml](./hello_world_flow.yaml)

## Initialization

In [ ]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_l0(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

In [ ]:
# Other imports
import os
from importlib import reload
from rs_common import prefect_utils
from rs_common.prefect_utils import *
import resources

In [ ]:
# We use only the eopf dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf

## Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [ ]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{OWNER_ID}/code" 

# Use a specific secret block on the bucket for this subfolder
code_bucket, os.environ["SHARE_BUCKET"] = await get_share_bucket(s3_code_folder)

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{code_bucket.bucket_name}/{code_bucket.bucket_folder}'")

# Upload local directory and resources contents
await code_bucket.put_directory(local_path = ".", to_path = ".")
await code_bucket.put_directory(local_path = resources.__path__[0], to_path = "resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = code_bucket.bucket_folder

In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./hello_world_flow.yaml"

In [ ]:
deploy_name = "hello-world/sprint19-hello-world"
await prefect_utils.wait_for_deployment(deploy_name)

## Run Prefect flow

In [ ]:
%%bash -s "$deploy_name"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --watch

NOTE: we could also call the Prefect flow from Python code. This is useful to debug.

In [ ]:
run_from_python = False
if run_from_python:
    # Import the module, or reload it if you changed its source code
    import hello_world_flow
    reload(hello_world_flow)
    
    # Run the flow
    results = hello_world_flow.hello_world()
    display(results)

## 3. Shutdown the dask clusters

In [ ]:
shutdown = False
if shutdown:
    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.